# Section 2 — Evolution across Location

We explore how the geographic distribution of names varies across French départements,
using three map-based visualisations.

In [1]:
import altair as alt
import pandas as pd
import geopandas as gpd

alt.data_transformers.enable('json')

names = pd.read_csv("Names hints/dpt2020.csv", sep=";")
names = names[names.preusuel != '_PRENOMS_RARES']
names = names[names.dpt != 'XX']
names['annais'] = pd.to_numeric(names['annais'], errors='coerce')
names = names.dropna(subset=['annais'])
names['annais'] = names['annais'].astype(int)
names['decade'] = (names['annais'] // 10) * 10

depts = gpd.read_file("Names hints/departements-version-simplifiee.geojson")

# Compute département centroids (Lambert 93 → WGS-84) — reused across all three visualisations
proj = depts.to_crs('EPSG:2154')
cents = proj.copy()
cents['geometry'] = proj.geometry.centroid
cents = cents.to_crs('EPSG:4326')
cents['lon'] = cents.geometry.x
cents['lat']  = cents.geometry.y
cent_df = pd.DataFrame(cents[['code', 'nom', 'lon', 'lat']])

# Pre-compute most popular name per (decade, dept) — used by 2.a and 2.b
by_ddd  = names.groupby(['decade', 'dpt', 'preusuel'])['nombre'].sum().reset_index()
totals  = names.groupby(['decade', 'dpt'])['nombre'].sum().reset_index(name='total')
top     = by_ddd.loc[by_ddd.groupby(['decade', 'dpt'])['nombre'].idxmax()].copy()
top     = top.merge(totals, on=['decade', 'dpt'])
top['proportion'] = top['nombre'] / top['total']
top['decade']     = top['decade'].astype(int)

names.head()

,sexe,preusuel,annais,dpt,nombre,decade
10885,1,AADIL,1983,84,3,1980
10886,1,AADIL,1992,92,3,1990
10888,1,AAHIL,2016,95,3,2010
10892,1,AARON,1962,75,3,1960
10893,1,AARON,1976,75,3,1970


## 2.a — Most popular name per département

Each département is filled according to the proportion its most-given name represents,
and the winning name is printed on top.
Drag the decade slider to see how regional dominance changes over time.

In [2]:
geo     = depts.merge(top, how='inner', left_on='code', right_on='dpt')
top_pos = cent_df.merge(top, left_on='code', right_on='dpt')

# Fixed domain across all decades; custom range so the minimum is a visible blue, not near-white
prop_range  = [top['proportion'].min(), top['proportion'].max()]
BLUE_SCALE  = alt.Scale(range=['#9ecae1', '#08306b'], domain=prop_range)

decade_param = alt.param(
    name='decade',
    bind=alt.binding_range(min=1900, max=2010, step=10, name='Decade: '),
    value=2000,
)

choropleth = (
    alt.Chart(geo)
    .mark_geoshape(stroke='white', strokeWidth=0.5)
    .encode(
        color=alt.Color('proportion:Q', scale=BLUE_SCALE, title='Top name proportion'),
        tooltip=['nom:N', 'preusuel:N', alt.Tooltip('proportion:Q', format='.2%')],
    )
    .transform_filter(alt.datum.decade == decade_param)
)

labels = (
    alt.Chart(top_pos)
    .mark_text(fontSize=7, align='center', baseline='middle', color='black')
    .encode(
        longitude='lon:Q',
        latitude='lat:Q',
        text='preusuel:N',
        tooltip=['nom:N', 'preusuel:N', alt.Tooltip('proportion:Q', format='.2%')],
    )
    .transform_filter(alt.datum.decade == decade_param)
)

(choropleth + labels).add_params(decade_param).project('mercator').properties(width=540, height=440)

alt.LayerChart(...)

## 2.b — Name word-cloud map

The most popular name for the selected decade is written directly on each département.
Font size is proportional to that name's share of local births.

In [3]:
text_data = cent_df.merge(top, left_on='code', right_on='dpt')

# Same fixed domain and colour range as 2.a
prop_range = [top['proportion'].min(), top['proportion'].max()]
BLUE_SCALE = alt.Scale(range=['#9ecae1', '#08306b'], domain=prop_range)

decade_param = alt.param(
    name='decade',
    bind=alt.binding_range(min=1900, max=2010, step=10, name='Decade: '),
    value=2000,
)

background = alt.Chart(depts).mark_geoshape(fill='#e8e8e8', stroke='white', strokeWidth=0.8)

labels = (
    alt.Chart(text_data)
    .mark_text(fontWeight='bold', align='center', baseline='middle')
    .encode(
        longitude='lon:Q',
        latitude='lat:Q',
        text='preusuel:N',
        size=alt.Size('proportion:Q', scale=alt.Scale(range=[7, 18]), legend=None),
        color=alt.Color('proportion:Q', scale=BLUE_SCALE, title='Proportion'),
        tooltip=['nom:N', 'preusuel:N', alt.Tooltip('proportion:Q', format='.2%')],
    )
    .transform_filter(alt.datum.decade == decade_param)
)

(background + labels).add_params(decade_param).project('mercator').properties(width=540, height=480)

alt.LayerChart(...)

## 2.c — Spatial distribution of a selected name

Use the dropdown to pick a name and the slider to choose a decade.
The colour is normalised per (name, decade): the darkest département always has the
highest local proportion for that name, so geographic concentration is visible even
for rare names.

In [4]:
per_decade = names.groupby(['decade', 'preusuel'])['nombre'].sum().reset_index()
per_decade['rank'] = (per_decade.groupby('decade')['nombre']
                                .rank(method='first', ascending=False)
                                .astype(int))
top30 = sorted(per_decade[per_decade['rank'] <= 30]['preusuel'].unique().tolist())

by_dn  = (names[names.preusuel.isin(top30)]
          .groupby(['decade', 'dpt', 'preusuel'])['nombre'].sum().reset_index())
totals = names.groupby(['decade', 'dpt'])['nombre'].sum().reset_index(name='total')
by_dn  = by_dn.merge(totals, on=['decade', 'dpt'])
by_dn['proportion'] = by_dn['nombre'] / by_dn['total']
by_dn['decade']     = by_dn['decade'].astype(int)

# Normalise per (name, decade) so every selection fills the full colour range
by_dn['prop_norm'] = by_dn.groupby(['preusuel', 'decade'])['proportion'].transform(
    lambda x: (x - x.min()) / (x.max() - x.min())
)

geo_dn = depts.merge(by_dn, how='inner', left_on='code', right_on='dpt')

decade_param = alt.param(
    name='decade',
    bind=alt.binding_range(min=1900, max=2010, step=10, name='Decade: '),
    value=2000,
)
name_param = alt.param(
    name='name',
    bind=alt.binding_select(options=top30, name='Name: '),
    value=top30[0],
)

alt.Chart(geo_dn).mark_geoshape(stroke='white', strokeWidth=0.5).encode(
    color=alt.Color(
        'prop_norm:Q',
        scale=alt.Scale(range=['#9ecae1', '#08306b'], domain=[0, 1]),
        title='Relative concentration',
    ),
    tooltip=['nom:N', 'preusuel:N', alt.Tooltip('proportion:Q', format='.3%')],
).transform_filter(
    (alt.datum.preusuel == name_param) & (alt.datum.decade == decade_param)
).add_params(decade_param, name_param).project('mercator').properties(width=540, height=440)

alt.Chart(...)